# RM571546 — Continuação do segundo checkpoint
Questões 4 a 8. As questões 1 a 3 já foram executadas no notebook original.

**Não execute novamente a carga de julho. Não use Run all.** Execute uma célula por vez e pare se houver erro ou falha de assert. Guarde o notebook original e suas capturas. Para entregar um único notebook, copie estas células para o fim do original; alternativamente, exporte os dois notebooks executados para posterior união preservando suas saídas.

Este arquivo contém código preparado, ainda não executado no Databricks. As saídas devem ser produzidas no seu workspace. Nenhuma resposta dissertativa ou captura foi fabricada.

Na questão 8, execute o UPDATE apenas uma vez e continue até RESTORE. Se algo falhar, não repita o UPDATE: envie o erro.


### 00 — Contexto e validação da Bronze
Execute esta célula na ordem indicada.

In [ ]:
from pyspark.sql import functions as F
from decimal import Decimal

spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA rm571546")

df_bronze = spark.table("workspace.rm571546.bronze_vendas")
assert df_bronze.count() == 58850, "Bronze diferente de 58.850 linhas; pare e envie o resultado."
assert len(df_bronze.columns) == 41, "Bronze deve ter 41 colunas."
print("Contexto: workspace.rm571546; Bronze validada com 58.850 linhas e 41 colunas.")

### Q4-A — Transformações, deduplicação e filtro
**CAPTURA: envie o código e o resultado completo desta célula.**

In [ ]:
colunas_prata = [
    "id_venda", "data", "sku", "produto_nome", "categoria", "marca",
    "id_loja", "loja_nome", "loja_cidade", "loja_uf", "loja_regiao",
    "id_cliente", "cliente_faixa_etaria", "cliente_cidade", "cliente_segmento",
    "quantidade", "valor_unitario", "desconto", "valor_venda", "canal_venda"
]

df_base = (
    df_bronze.select(*colunas_prata)
    .withColumn("data", F.to_date("data"))
    .withColumn("loja_uf", F.upper(F.trim(F.col("loja_uf"))))
    .withColumn("valor_venda", F.col("valor_venda").cast("decimal(12,2)"))
    .withColumn("canal_venda", F.coalesce(F.col("canal_venda"), F.lit("Loja Física")))
)

# A ordem exigida é deduplicar primeiro e filtrar depois.
df_sem_dup = df_base.dropDuplicates(["id_venda"])
df_prata = df_sem_dup.filter(F.col("valor_venda") > 0)

n_bronze = df_bronze.count()
n_sem_dup = df_sem_dup.count()
n_prata = df_prata.count()

resumo_qualidade = spark.createDataFrame([
    (1, "Linhas na bronze_vendas", n_bronze),
    (2, "Linhas após remover duplicidades", n_sem_dup),
    (3, "Linhas removidas por duplicidade", n_bronze - n_sem_dup),
    (4, "Linhas removidas por valor inválido", n_sem_dup - n_prata),
    (5, "Linhas na prata_vendas", n_prata),
], ["ordem", "etapa", "quantidade_linhas"])

display(resumo_qualidade.orderBy("ordem").drop("ordem"))
assert len(df_prata.columns) == 20
assert df_prata.filter(F.col("data").isNull()).count() == 0

df_prata.createOrReplaceTempView("prata_preparada_temp")
spark.sql("""
    CREATE OR REPLACE TABLE workspace.rm571546.prata_vendas
    USING DELTA AS SELECT * FROM prata_preparada_temp
""")
print(f"Prata gravada: {n_prata} linhas e {len(df_prata.columns)} colunas.")

### Q4-B — Sete UFs
**CAPTURA: envie o código e o resultado completo desta célula.**

In [ ]:
%sql
SELECT DISTINCT loja_uf
FROM workspace.rm571546.prata_vendas
ORDER BY 1;

### Q4-C — Schema e validação da Prata
**CAPTURA: envie o código e o resultado completo desta célula.**

In [ ]:
df_prata_salva = spark.table("workspace.rm571546.prata_vendas")
df_prata_salva.printSchema()
assert df_prata_salva.select("loja_uf").distinct().count() == 7
assert df_prata_salva.select("id_venda").distinct().count() == df_prata_salva.count()
print("Validação concluída: 20 colunas, 7 UFs e id_venda sem duplicidade.")

### Q5-A — Criar cinco dimensões e a fato
**CAPTURA: envie o código e o resultado completo desta célula.**

In [ ]:
df_p = spark.table("workspace.rm571546.prata_vendas")

tabelas_ouro = {
    "dim_produto": df_p.select("sku", "produto_nome", "categoria", "marca").distinct(),
    "dim_loja": df_p.select("id_loja", "loja_nome", "loja_cidade", "loja_uf", "loja_regiao").distinct(),
    "dim_cliente": df_p.select("id_cliente", "cliente_faixa_etaria", "cliente_cidade", "cliente_segmento").distinct(),
    "dim_tempo": df_p.select("data").distinct()
        .withColumn("ano", F.year("data"))
        .withColumn("trimestre", F.quarter("data"))
        .withColumn("mes", F.month("data")),
    "dim_canal": df_p.select("canal_venda").distinct(),
    "fato_vendas": df_p.select(
        "id_venda", "data", "sku", "id_loja", "id_cliente", "canal_venda",
        "quantidade", "valor_unitario", "desconto", "valor_venda"
    )
}

for nome, dataframe in tabelas_ouro.items():
    dataframe.createOrReplaceTempView("ouro_preparado_temp")
    spark.sql(f"CREATE OR REPLACE TABLE workspace.rm571546.{nome} USING DELTA AS SELECT * FROM ouro_preparado_temp")
    print(f"Tabela criada: {nome}")

### Q5-B — Quantidade de linhas das dimensões
**CAPTURA: envie o código e o resultado completo desta célula.**

In [ ]:
%sql
SELECT 'dim_produto' AS tabela, COUNT(*) AS quantidade_linhas FROM workspace.rm571546.dim_produto
UNION ALL
SELECT 'dim_loja', COUNT(*) FROM workspace.rm571546.dim_loja
UNION ALL
SELECT 'dim_cliente', COUNT(*) FROM workspace.rm571546.dim_cliente
UNION ALL
SELECT 'dim_tempo', COUNT(*) FROM workspace.rm571546.dim_tempo
UNION ALL
SELECT 'dim_canal', COUNT(*) FROM workspace.rm571546.dim_canal;

### Q5-C — Checagem de chave duplicada de loja
**CAPTURA: envie o código e o resultado completo desta célula.**

In [ ]:
%sql
SELECT id_loja, COUNT(*) AS quantidade
FROM workspace.rm571546.dim_loja
GROUP BY id_loja
HAVING COUNT(*) > 1;

### Q5-D — Integridade e faturamento de referência
**CAPTURA: envie o código e o resultado completo desta célula.**

In [ ]:
soma_prata = spark.table("workspace.rm571546.prata_vendas").agg(F.sum("valor_venda").alias("total")).first()["total"]
soma_fato = spark.table("workspace.rm571546.fato_vendas").agg(F.sum("valor_venda").alias("total")).first()["total"]

display(spark.createDataFrame([
    ("prata_vendas", soma_prata),
    ("fato_vendas", soma_fato)
], "tabela string, faturamento decimal(20,2)"))

assert soma_prata == soma_fato, "Faturamentos diferentes: pare antes de continuar."
assert spark.table("workspace.rm571546.dim_loja").count() == 40
total_referencia = soma_fato
print(f"Integridade confirmada. Faturamento de referência para Q8: {total_referencia}")

### Q5-E — Dimensão de loja sem tratamento da UF (sem gravar)
**CAPTURA: envie o código e o resultado completo desta célula.**

In [ ]:
%sql
SELECT COUNT(*) AS lojas_sem_tratamento
FROM (
    SELECT DISTINCT id_loja, loja_nome, loja_cidade, loja_uf, loja_regiao
    FROM workspace.rm571546.bronze_vendas
) lojas;

### Q6-A — Evolução mensal por região com LAG
**CAPTURA: envie o código e o resultado completo desta célula.**

In [ ]:
%sql
WITH mensal AS (
    SELECT l.loja_regiao, t.ano, t.mes,
           SUM(f.valor_venda) AS faturamento
    FROM workspace.rm571546.fato_vendas f
    JOIN workspace.rm571546.dim_loja l ON f.id_loja = l.id_loja
    JOIN workspace.rm571546.dim_tempo t ON f.data = t.data
    WHERE t.ano = 2026 AND t.mes BETWEEN 1 AND 7
    GROUP BY l.loja_regiao, t.ano, t.mes
), comparacao AS (
    SELECT *,
           LAG(faturamento) OVER (
               PARTITION BY loja_regiao ORDER BY ano, mes
           ) AS faturamento_anterior
    FROM mensal
)
SELECT loja_regiao, mes, faturamento, faturamento_anterior,
       ROUND((faturamento - faturamento_anterior)
             / NULLIF(faturamento_anterior, 0) * 100, 2) AS variacao_pct
FROM comparacao
ORDER BY loja_regiao, ano, mes;

Q6-A: envie todas as linhas, em mais de uma captura se necessário. Também exporte o resultado em CSV pelo botão de download da tabela, se disponível. Isso evita perder números na interpretação.

### Q6-B — Quantos dias estão representados em cada mês
**CAPTURA: envie o código e o resultado completo desta célula.**

In [ ]:
%sql
SELECT ano, mes, COUNT(DISTINCT data) AS dias_presentes,
       DAY(LAST_DAY(MIN(data))) AS dias_calendario,
       MIN(data) AS primeira_data, MAX(data) AS ultima_data
FROM workspace.rm571546.dim_tempo
WHERE ano = 2026
GROUP BY ano, mes
ORDER BY ano, mes;

### Q7-A — Faturamento, ticket e participação dos canais em julho
**CAPTURA: envie o código e o resultado completo desta célula.**

In [ ]:
%sql
SELECT c.canal_venda,
       SUM(f.valor_venda) AS faturamento,
       COUNT(DISTINCT f.id_venda) AS vendas,
       ROUND(SUM(f.valor_venda) / COUNT(DISTINCT f.id_venda), 2) AS ticket_medio,
       ROUND(100 * SUM(f.valor_venda)
             / SUM(SUM(f.valor_venda)) OVER (), 2) AS participacao_pct,
       ROUND(100.0 * COUNT(DISTINCT f.id_venda)
             / SUM(COUNT(DISTINCT f.id_venda)) OVER (), 2) AS participacao_vendas_pct
FROM workspace.rm571546.fato_vendas f
JOIN workspace.rm571546.dim_canal c ON f.canal_venda = c.canal_venda
JOIN workspace.rm571546.dim_tempo t ON f.data = t.data
WHERE t.ano = 2026 AND t.mes = 7
GROUP BY c.canal_venda
ORDER BY faturamento DESC;

## Q8 — Execute uma vez e em sequência até a restauração
Não recrie a fato entre estas células. Não repita o UPDATE se ocorrer erro em uma etapa posterior. A versão correta será obtida do histórico, sem assumir versão 0.

### Q8-A — Proteção e registro do estado antes do erro
**CAPTURA: envie o código e o resultado completo desta célula.**

In [ ]:
historico_antes = spark.sql("DESCRIBE HISTORY workspace.rm571546.fato_vendas")
assert historico_antes.filter(F.col("operation") == "UPDATE").count() == 0, (
    "Já existe UPDATE no histórico. Não repita o erro; envie o histórico para análise."
)
versao_antes_update = historico_antes.agg(F.max("version")).first()[0]
total_referencia = spark.table("workspace.rm571546.prata_vendas").agg(F.sum("valor_venda")).first()[0]
total_antes_update = spark.table("workspace.rm571546.fato_vendas").agg(F.sum("valor_venda")).first()[0]
total_julho_antes = spark.sql("""
    SELECT SUM(valor_venda) FROM workspace.rm571546.fato_vendas
    WHERE data >= '2026-07-01'
""").first()[0]
assert total_antes_update == total_referencia
print(f"Versão antes do UPDATE: {versao_antes_update}")
print(f"Faturamento correto: {total_referencia}")
print(f"Faturamento de julho antes do erro: {total_julho_antes}")

### Q8-B — Reproduzir o erro (UMA ÚNICA VEZ)
**CAPTURA: envie o código e o resultado completo desta célula.**

In [ ]:
%sql
UPDATE fato_vendas SET valor_venda = valor_venda * 100 WHERE data >= '2026-07-01';

### Q8-C — Soma após o erro
**CAPTURA: envie o código e o resultado completo desta célula.**

In [ ]:
%sql
SELECT SUM(valor_venda) AS faturamento_apos_erro
FROM workspace.rm571546.fato_vendas;

### Q8-D — Histórico após o UPDATE
**CAPTURA: envie o código e o resultado completo desta célula.**

In [ ]:
%sql
DESCRIBE HISTORY workspace.rm571546.fato_vendas;

### Q8-E — Identificar a versão anterior e consultar Time Travel
**CAPTURA: envie o código e o resultado completo desta célula.**

In [ ]:
historico = spark.sql("DESCRIBE HISTORY workspace.rm571546.fato_vendas")
versao_update = historico.filter(F.col("operation") == "UPDATE").agg(F.max("version")).first()[0]
assert versao_update is not None, "Nenhum UPDATE encontrado no histórico."
versao_correta = historico.filter(F.col("version") < versao_update).agg(F.max("version")).first()[0]
assert versao_correta is not None
assert versao_correta == versao_antes_update, "Histórico mudou inesperadamente; envie as capturas."

total_apos_erro = spark.table("workspace.rm571546.fato_vendas").agg(F.sum("valor_venda")).first()[0]
assert total_apos_erro == total_referencia + Decimal(99) * total_julho_antes

consulta_time_travel = f"""
    SELECT SUM(valor_venda) AS faturamento_na_versao_correta
    FROM workspace.rm571546.fato_vendas VERSION AS OF {versao_correta}
"""
print(f"UPDATE na versão {versao_update}; versão correta anterior: {versao_correta}")
print(consulta_time_travel)
resultado_time_travel = spark.sql(consulta_time_travel)
display(resultado_time_travel)
assert resultado_time_travel.first()[0] == total_referencia

### Q8-F — Restaurar a versão identificada
**CAPTURA: envie o código e o resultado completo desta célula.**

In [ ]:
comando_restore = f"RESTORE TABLE workspace.rm571546.fato_vendas TO VERSION AS OF {versao_correta}"
print(comando_restore)
display(spark.sql(comando_restore))

### Q8-G — Confirmar que o faturamento foi recuperado
**CAPTURA: envie o código e o resultado completo desta célula.**

In [ ]:
total_restaurado = spark.table("workspace.rm571546.fato_vendas").agg(F.sum("valor_venda")).first()[0]
display(spark.createDataFrame([
    ("Referência da questão 5b", total_referencia),
    ("Fato após RESTORE", total_restaurado)
], "etapa string, faturamento decimal(20,2)"))
assert total_restaurado == total_referencia, "O faturamento ainda não foi recuperado."
print("RESTORE confirmado: faturamento igual ao da questão 5b.")

### Q8-H — Histórico final com RESTORE
**CAPTURA: envie o código e o resultado completo desta célula.**

In [ ]:
%sql
DESCRIBE HISTORY workspace.rm571546.fato_vendas;

## O que enviar para montar o documento final
- Capturas Q4-A, Q4-B e Q4-C.
- Capturas Q5-A, Q5-B, Q5-C (resultado vazio também), Q5-D e Q5-E.
- Capturas de todas as linhas de Q6-A e de Q6-B; se possível, CSV de Q6-A.
- Captura Q7-A com todas as colunas.
- Capturas Q8-A até Q8-H, incluindo código do UPDATE, soma incorreta, histórico, VERSION AS OF, RESTORE e soma recuperada.
- Nome completo de cada integrante, respectivos RMs e quem faz a entrega.
- Notebook(s) exportado(s) como IPython Notebook depois de executar. O original contém as evidências das questões 1 a 3.
- Captura das contagens 50.000 linhas e 40 colunas da questão 2, que estava fora da área visível na captura anterior.

Capture código e resultado legíveis. Não recorte mensagens de erro importantes. Não clique em Run all no notebook original: ele contém uma tentativa de erro esperada e uma carga de julho que não deve se repetir.
